In [110]:
import sys
sys.path.append('../_amigocloud')

from sqlalchemy import create_engine, text
from amigocloud import AmigoCloud
from docx import Document
from docx2pdf import convert
import os
import json
import pandas as pd

# CONEXIÓN A GOOGLE DRIVE
from pydrive.auth import GoogleAuth
from pydrive.drive import GoogleDrive
import os

gauth = GoogleAuth()

# Si existe el archivo, intenta cargarlo
if os.path.exists("mycreds.txt"):
    gauth.LoadCredentialsFile("mycreds.txt")

# Si no hay credenciales o son inválidas, forzar nueva autenticación
if gauth.credentials is None or gauth.access_token_expired:
    try:
        gauth.LocalWebserverAuth()
    except:
        print("Error de autenticación. Eliminá 'mycreds.txt' y volvé a intentarlo.")
else:
    gauth.Authorize()

gauth.SaveCredentialsFile("mycreds.txt")
drive = GoogleDrive(gauth)

In [111]:
sys.path.append('..')

from config import RUTA_UNIDAD_ONE_DRIVE
from config import RUTA_LOCAL_ONE_DRIVE
from config import POSTGRES_UTEA
RUTA_COMPLETA = os.path.join(RUTA_UNIDAD_ONE_DRIVE, RUTA_LOCAL_ONE_DRIVE)

In [112]:
def get_ruta_informes():
    try:
        with open('../../ruta_informes.txt', 'r') as file:
            ruta_inf = file.readline().strip()
            return ruta_inf
    except FileNotFoundError:
        print("El archivo 'ruta_local.txt' no se encuentra en la ruta especificada.")
    return None

In [113]:
ruta_informes = get_ruta_informes()
directorio_origen = ruta_informes + r'\_informes'
directorio_destino = ruta_informes + r'\_pdf'

In [114]:
# Recorre y convierte a pdf archivos .docx que no empiezan con "_"
def convertir_docx_to_pdf():
    for archivo in os.listdir(directorio_origen):
        #if archivo.lower().endswith(".docx") and not archivo.startswith("_"):
        if True:
            ruta_docx = os.path.join(directorio_origen, archivo)
            ruta_pdf = os.path.join(directorio_destino, archivo.replace(".docx", ".pdf"))
            
            try:
                convert(ruta_docx, ruta_pdf)
                print(ruta_docx)
                if os.path.exists(ruta_pdf):
                    print(f"✅ Convertido: {archivo}")
                    os.remove(ruta_docx)
                    print(f"🗑️ Eliminado original: {archivo}")
                else:
                    print(f"❌ Error: PDF no generado para {archivo}")
            except Exception as e:
                print(f"⚠️ Error al convertir {archivo}: {e}")

def obtener_engine():
    return create_engine(
        f"postgresql+psycopg2://{POSTGRES_UTEA['USER']}:{POSTGRES_UTEA['PASSWORD']}@{POSTGRES_UTEA['HOST']}:{POSTGRES_UTEA['PORT']}/{POSTGRES_UTEA['DATABASE']}"
    )

def cargar_link_to_db(url, cite):
    engine = obtener_engine()
    try:
        with engine.begin() as conn:
            query = text("""
                UPDATE drones_control_bio.resumen_control_bio SET link = :url WHERE cite = :cite
            """)
            conn.execute(query, {"url": url, "cite": cite})
            print(f"✔️ Se registro link para CITE: {cite}" )
    except Exception as e:
        print(f"❌ Error al actualizar CITE: {cite}, {e}")

def get_registros_sin_enviar():
    engine = obtener_engine()
    try:
        query = """
            SELECT *
            FROM drones_control_bio.resumen_control_bio
            WHERE enviado = false
        """
        df = pd.read_sql(query, engine)
        return df
    except Exception as e:
        print(f"❌ Error al consultar unidades sin labor: {e}")
        return pd.DataFrame()
    return None

In [115]:
convertir_docx_to_pdf()

  0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\Usuario\Desktop\PROYECTOS\_informes\10380_SLCB_24-10-2026_PAICHUCAMA CACERES JOSE FELIX_CB-2025-368.docx
✅ Convertido: 10380_SLCB_24-10-2026_PAICHUCAMA CACERES JOSE FELIX_CB-2025-368.docx
🗑️ Eliminado original: 10380_SLCB_24-10-2026_PAICHUCAMA CACERES JOSE FELIX_CB-2025-368.docx


  0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\Usuario\Desktop\PROYECTOS\_informes\11231_SLCB_24-10-2026_RAFAEL GARNICA FIDEL_CB-2025-384.docx
✅ Convertido: 11231_SLCB_24-10-2026_RAFAEL GARNICA FIDEL_CB-2025-384.docx
🗑️ Eliminado original: 11231_SLCB_24-10-2026_RAFAEL GARNICA FIDEL_CB-2025-384.docx


  0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\Usuario\Desktop\PROYECTOS\_informes\1180_SLCB_24-10-2026_BALCAZAR BALCAZAR JESUS E._CB-2025-375.docx
✅ Convertido: 1180_SLCB_24-10-2026_BALCAZAR BALCAZAR JESUS E._CB-2025-375.docx
🗑️ Eliminado original: 1180_SLCB_24-10-2026_BALCAZAR BALCAZAR JESUS E._CB-2025-375.docx


... (36 outputs left out)

  0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\Usuario\Desktop\PROYECTOS\_informes\7846_SLCB_24-10-2026_LINO SORIA MANFREDO_CB-2025-383.docx
✅ Convertido: 7846_SLCB_24-10-2026_LINO SORIA MANFREDO_CB-2025-383.docx
🗑️ Eliminado original: 7846_SLCB_24-10-2026_LINO SORIA MANFREDO_CB-2025-383.docx


  0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\Usuario\Desktop\PROYECTOS\_informes\9722_SLCB_03-09-2026_OLGUIN FLORES MIGUELINA_CB-2025-381.docx
✅ Convertido: 9722_SLCB_03-09-2026_OLGUIN FLORES MIGUELINA_CB-2025-381.docx
🗑️ Eliminado original: 9722_SLCB_03-09-2026_OLGUIN FLORES MIGUELINA_CB-2025-381.docx


In [116]:
def cargar_pdf_to_drive(id_drive, title_file, path_file):
    try:
        # 1. Buscar si ya existe un archivo con el mismo nombre en la misma carpeta
        query = f"'{id_drive}' in parents and title = '{title_file}' and trashed = false"
        file_list = drive.ListFile({'q': query}).GetList()

        if file_list:
            # Si ya existe, usamos el primero encontrado
            file = file_list[0]
            print(f"[INFO] El archivo '{title_file}' ya existe. Se reemplazará.")
        else:
            # Si no existe, creamos uno nuevo
            file = drive.CreateFile({
                "mimeType": "application/pdf",
                "parents": [{"kind": "drive#fileLink", "id": id_drive}]
            })
            file['title'] = title_file
            print(f"[INFO] El archivo '{title_file}' no existe. Se creará uno nuevo.")

        # 2. Reemplazar el contenido
        file.SetContentFile(path_file)
        file.Upload()

        # 3. Dar permisos públicos si es nuevo
        if not file_list:
            file.InsertPermission({
                'type': 'anyone',
                'value': 'anyone',
                'role': 'reader'
            })

        print(f"✅[OK] Archivo '{title_file}' subido exitosamente.")
        return file['alternateLink']

    except Exception as e:
        print(f"❌[ERROR] No se pudo subir el archivo '{title_file}': {e}")
        return False

In [117]:
contenido = os.listdir(directorio_destino)
print(len(contenido))
contenido

23


['10380_SLCB_24-10-2026_PAICHUCAMA CACERES JOSE FELIX_CB-2025-368.pdf',
 '11231_SLCB_24-10-2026_RAFAEL GARNICA FIDEL_CB-2025-384.pdf',
 '1180_SLCB_24-10-2026_BALCAZAR BALCAZAR JESUS E._CB-2025-375.pdf',
 '12998_SLCB_24-10-2026_SUAREZ REA MARCO ANTONIO_CB-2025-370.pdf',
 '1741_SLCB_24-10-2026_CARO ALTAMIRANO ROSARIO_CB-2025-369.pdf',
 '1741_SLCB_24-10-2026_CARO ALTAMIRANO ROSARIO_CB-2025-385.pdf',
 '17680_SLCB_24-10-2026_BALDELOMAR HIDALGO JULIETA_CB-2025-372.pdf',
 '18028_SLCB_22-10-2026_RIVERO GUTIERREZ FRANCISCO_CB-2025-362.pdf',
 '18189_SLCB_24-10-2026_BALCAZAR SOLETO RAMON BERTHY_CB-2025-386.pdf',
 '20055_SLCB_24-10-2026_FRANCO ARIAS EDGAR_CB-2025-382.pdf',
 '2658_SLCB_22-10-2026_CLAURE OVANDO HILARION_CB-2025-367.pdf',
 '2899_SLCB_24-10-2026_EGUEZ ELHAGE JOSE EDUARDO_CB-2025-371.pdf',
 '3378_SLCB_24-10-2026_ESCOBAR MORATO VICTOR HUGO_CB-2025-374.pdf',
 '3494_SLCB_24-10-2026_ESCOBAR VEIZAGA MIGUEL ANDRES_CB-2025-379.pdf',
 '40158_SLCB_22-10-2026_GUZMAN CACERES MARIA LOURDES_CB-2025

In [118]:
for file in contenido:
    partes = file.split('_')
    
    cite = partes[-1].split('.')[0]
    sigla = partes[1]
    title_file = file
    id_drive = '1sEfhWGATb7tkKgr6gcknH3ZA_ed_hGTq'
    path_file = os.path.join(directorio_destino, file)
    print(cite)
    url_drive = cargar_pdf_to_drive(id_drive, title_file, path_file)
    if url_drive == False:
        continue
    cargar_link_to_db(url_drive, cite)
    if os.path.exists(path_file):
        os.remove(path_file)
        print("✅ Archivo eliminado.")
    else:
        print("⚠️ El archivo no existe.")

CB-2025-368
[INFO] El archivo '10380_SLCB_24-10-2026_PAICHUCAMA CACERES JOSE FELIX_CB-2025-368.pdf' no existe. Se creará uno nuevo.
✅[OK] Archivo '10380_SLCB_24-10-2026_PAICHUCAMA CACERES JOSE FELIX_CB-2025-368.pdf' subido exitosamente.
✔️ Se registro link para CITE: CB-2025-368
✅ Archivo eliminado.
CB-2025-384
[INFO] El archivo '11231_SLCB_24-10-2026_RAFAEL GARNICA FIDEL_CB-2025-384.pdf' no existe. Se creará uno nuevo.
✅[OK] Archivo '11231_SLCB_24-10-2026_RAFAEL GARNICA FIDEL_CB-2025-384.pdf' subido exitosamente.
✔️ Se registro link para CITE: CB-2025-384
✅ Archivo eliminado.
CB-2025-375
[INFO] El archivo '1180_SLCB_24-10-2026_BALCAZAR BALCAZAR JESUS E._CB-2025-375.pdf' no existe. Se creará uno nuevo.
✅[OK] Archivo '1180_SLCB_24-10-2026_BALCAZAR BALCAZAR JESUS E._CB-2025-375.pdf' subido exitosamente.
✔️ Se registro link para CITE: CB-2025-375
✅ Archivo eliminado.
CB-2025-370
[INFO] El archivo '12998_SLCB_24-10-2026_SUAREZ REA MARCO ANTONIO_CB-2025-370.pdf' no existe. Se creará uno nue

In [23]:
import psycopg2

In [25]:
def obtener_conexion():
    return psycopg2.connect(
        host='localhost', 
        port=5433, 
        database='utea', 
        user='postgres', 
        password='A123456*')
    
def insertar_mensaje_whatsapp(cod_canero, nombre_canero, numero_contac, mensaje, enviado=False, fecha_enviado=None):
    try:
        conexion = obtener_conexion()
        cursor = conexion.cursor()
        cursor.execute("""
            INSERT INTO notificaciones_wsp.msj_whatsapp (
                cod_canero, nombre_canero, numero_cel, mensaje, enviado, fecha_enviado, numero_contac
            ) VALUES (%s, %s, %s, %s, %s, %s, %s)
        """, (cod_canero, nombre_canero, 0, mensaje, enviado, fecha_enviado, numero_contac))
        
        conexion.commit()
        cursor.close()
        print("Mensaje insertado correctamente.")
    except Exception as e:
        conexion.rollback()
        print("Error inesperado")
    return None

def marcar_resumen_lib_enviado(id_resumen):
    try:
        conexion = obtener_conexion()
        cursor = conexion.cursor()
        cursor.execute("""
            UPDATE drones_control_bio.resumen_control_bio
            SET enviado = true
            WHERE id = %s
        """, (id_resumen,))
        conexion.commit()
        cursor.close()
        print(f"Mensaje con id={id_resumen} marcado como enviado.")
    except Exception as e:
        conexion.rollback()
        print(f"Error inesperado: {e}")
    finally:
        if conexion:
            conexion.close()
    return None

In [26]:
PATH_XLSX_CONTAC=r'G:\\Ingenio Azucarero Guabira S.A\\UTEA - SEMANAL - EQUIPO AVIACION UTEA\\Pulverizacion\\Data\\CONTACTOS.xlsx'

In [27]:
sin_enviar = get_registros_sin_enviar()

In [28]:
cods_ca = list(set(sin_enviar['id']))
print(cods_ca)

[740, 741, 742, 743, 744, 745, 746, 747, 748, 749, 750, 751, 752, 753, 754, 755, 756, 757, 758, 759, 760, 761, 762, 763, 764, 765, 766, 767, 768, 769, 770, 771, 772, 773, 776, 777, 778, 779, 780, 781, 782, 783, 784, 785, 786, 787, 788, 789, 790, 791, 792, 793, 794, 795, 796, 797, 798, 799, 825, 826, 827, 828, 829, 830, 831, 832, 833, 834, 835, 836, 837, 838, 839, 840, 841, 842, 843, 844, 845, 846, 847, 848, 849, 850, 851, 852, 853, 854, 855, 889, 890, 891, 892, 893, 894, 895, 896, 897, 898, 899, 900, 901, 902, 903, 904, 905, 906, 907, 908, 909, 910, 911, 912, 913, 914, 915, 916, 917, 918, 919, 920, 921, 922, 962, 963, 964, 965, 966, 967, 968, 969, 970, 971, 972, 973, 974, 975, 976, 977, 978, 979, 980, 981, 982, 983, 984, 985, 986, 987, 988, 989, 990, 991, 992, 993, 994, 995, 996, 997, 998, 999, 1000, 1001, 1002, 1004, 1005, 1006, 1007, 1008, 1009, 1010, 1011, 1012, 1013, 1014, 1015]


In [29]:
len(cods_ca)

176

In [20]:
df_contac = pd.read_excel(PATH_XLSX_CONTAC)
df_contac.fillna(0, inplace=True)
df_contac = df_contac[['cod_ca', 'nom_ca', 'telf01', 'telf02', 'telf03']]

In [21]:
df_contac.head(3)

,cod_ca,nom_ca,telf01,telf02,telf03
0,86,AGUILERA TARADELLES JOSE LUIS,67759316,77010520.0,68848905.0
1,42239,CORDOVA OLGUIN LEONARDO,78437581,74694116.0,69005644.0
2,30035,GUTIERREZ AGUILERA WIDEN GUILLER,76049459,69111101.0,69111108.0


In [22]:
sin_enviar.head(3)

,id,fecha,institucion,cod_canero,nombre_canero,hectareas,cite,cod_propiedad,propiedad,producto,link,agrocittca,enviado
0,738,2026-06-26,0,1530,C.I.T.T.C.A.,225.18,CB-2025-14,299,CITTCA,['CRISOPA'],https://drive.google.com/file/d/1zxsW81IE4XOmQ...,True,False
1,739,2026-06-30,0,1530,C.I.T.T.C.A.,96.36,CB-2025-15,299,CITTCA,['CRISOPA'],https://drive.google.com/file/d/1sGHwUkkO2Slyk...,True,False
2,740,2026-07-24,0,42667,PESO AGROPECUARIA S.R.L.,30.91,CB-2025-16,2358,HERRADURA--PESO,['CRISOPA'],https://drive.google.com/file/d/1zk43pghb0beaN...,True,False


### Mensaje dirigiendo a agrocittca

In [19]:
for index, row in sin_enviar.iterrows():    
    contac_filtro = df_contac[df_contac['cod_ca']==row['cod_canero']]
    if not contac_filtro.empty:  # Verifica si hay al menos una fila
        contac = contac_filtro.iloc[0]
        telefonos = [int(contac['telf01']), int(contac['telf02']), int(contac['telf03'])]
        telefonos_validos = [telefono for telefono in telefonos if len(str(telefono)) == 8]
        if (len(telefonos_validos) == 0):
            print(f'❌ Codigo: {cod} no tinene numeros validos registrados')
            continue
        
        for num in telefonos_validos:
            msj = f'''🌱 *Saludos {row['nombre_canero']}*, le escribimos desde *UTEA - GUABIRÁ* para comunicarle que el informe del último servicio de *🐝 Control Biológico* realizado ya se encuentra disponible en *AgroCITTCA*.

📲 *Adjuntamos sus credenciales* para acceder a la App y los enlaces de descarga 👇

    👤 *Usuario:* {row['cod_canero']}
    🔑 *Contraseña:* 123*{row['cod_canero']}

⬇️ *Descargue la App:* 

📱 *Android:* https://play.google.com/store/apps/details?id=com.dima.guabira  

🍎 *iPhone:* https://apps.apple.com/us/app/agro-cittca-guabir%C3%A1/id1669149924

🚀 Una vez instalada la app, inicie sesión con su usuario y contraseña, y acceda a:  
📂 *AGROCITTCA → Seguimiento a campo → Informes*

Si tiene inconvenientes para acceder o visualizar los informes, no dude en consultarnos. ¡Gracias por su confianza! 🙌
'''         
            insertar_mensaje_whatsapp(
                cod_canero=row['cod_canero'],
                nombre_canero=row['nombre_canero'],
                numero_contac=f'591{num}@s.whatsapp.net',
                mensaje=msj
            )
            marcar_resumen_lib_enviado(row['id'])
    else:
        print(f"Codigo: {row['cod_canero']} - {row['nombre_canero']} no tiene registro telefonico")

Codigo: 1165 - BEJARANO GANDARILLA ERWIN no tiene registro telefonico
Codigo: 1478 - CALLE RALDEZ MARCO ANTONIO no tiene registro telefonico
Codigo: 7506 - LOHENER ROSENDA SHEILA PAZ DE no tiene registro telefonico
Codigo: 41562 - SUAREZ SALAS ERWIN RONALD no tiene registro telefonico
Codigo: 1182 - BOHORQUEZ FERNANDEZ NICANOR no tiene registro telefonico
Codigo: 2353 - CARO BENITES GERMAN no tiene registro telefonico
Codigo: 13638 - TISCO VASQUEZ SERGIO no tiene registro telefonico
Codigo: 1182 - BOHORQUEZ FERNANDEZ NICANOR no tiene registro telefonico
Codigo: 13638 - TISCO VASQUEZ SERGIO no tiene registro telefonico
Codigo: 2351 - CANDIA RUIZ ADALBERTO no tiene registro telefonico
Codigo: 2353 - CARO BENITES GERMAN no tiene registro telefonico
Codigo: 42378 - CANDIA RUIZ ORLANDO no tiene registro telefonico
Codigo: 1478 - CALLE RALDEZ MARCO ANTONIO no tiene registro telefonico
Codigo: 1165 - BEJARANO GANDARILLA ERWIN no tiene registro telefonico
Codigo: 42229 - CANDIA ROJAS JOSE LUIS

### Mensaje dirigiendo a PDF

In [23]:
for index, row in sin_enviar.iterrows():    
    contac_filtro = df_contac[df_contac['cod_ca']==row['cod_canero']]
    if not contac_filtro.empty:  # Verifica si hay al menos una fila
        contac = contac_filtro.iloc[0]
        telefonos = [int(contac['telf01']), int(contac['telf02']), int(contac['telf03'])]
        telefonos_validos = [telefono for telefono in telefonos if len(str(telefono)) == 8]
        if (len(telefonos_validos) == 0):
            print(f'Codigo: {cod} no tinene numeros validos registrados')
            continue
        
        for num in telefonos_validos:
            msj = f"Saludos *{row['nombre_canero']}*, le escribimos desde *UTEA - GUABIRA* para hacer llegar una copia del informe de servicio de control biologico en caña de azucar. También informale este informe lo puede encotrar en la aplicación AgroCITTCA, y que a partir de los proximos servicos los informes se enviaran direntamente a AgroCITTCA. Adjunto informe: {row['link']}"
            insertar_mensaje_whatsapp(
                cod_canero=row['cod_canero'],
                nombre_canero=row['nombre_canero'],
                numero_contac=f'591{num}@s.whatsapp.net',
                mensaje=msj
            )
            marcar_resumen_lib_enviado(row['id'])
    else:
        print(f"Codigo: {row['cod_canero']} - {row['nombre_canero']} no tiene registro telefonico")

Mensaje insertado correctamente.
Mensaje con id=738 marcado como enviado.
Mensaje insertado correctamente.
Mensaje con id=738 marcado como enviado.
Mensaje insertado correctamente.
Mensaje con id=738 marcado como enviado.
Mensaje insertado correctamente.
Mensaje con id=739 marcado como enviado.
Mensaje insertado correctamente.
Mensaje con id=739 marcado como enviado.
Mensaje insertado correctamente.
Mensaje con id=739 marcado como enviado.
Codigo: 42667 - PESO AGROPECUARIA S.R.L. no tiene registro telefonico
